In [5]:
import json

In [6]:
raw_dataset_path = "./text2MongoDB_dataset/dataset.json"
data_save_path = "./text2MongoDB_dataset/dataset_final.json"
MQSpider = "./text2MongoDB_dataset/MQSpider.json"

In [7]:
with open(raw_dataset_path, "r") as f:
    data_all = json.load(f)

with open(MQSpider, "r") as f:
    mqlspider = json.load(f)

mql_spider = {}

for example in mqlspider:
    mql_spider[example['record_id']] = example


In [9]:

data_new = []
for id, example in enumerate(data_all):
    
    db_id = example['db_id']
    nlqs = example['question']
    ref_sql = example['query']
    try:
        example_new = {
            "record_id":id,
            "db_id":db_id,
            "nl_queries":nlqs,
            "ref_sql":ref_sql,
            "mql_nodebug":mql_spider[id]['mql_nodebug'],
            "mql_debugged":mql_spider[id]['mql_debugged'],
            "info":mql_spider[id]['info']
        }
    except:
        example_new = {
            "record_id":id,
            "db_id":db_id,
            "nl_queries":nlqs,
            "ref_sql":ref_sql,
            "mql_nodebug":"",
            "mql_debugged":"",
            "info":{"match":"", "info":""}
        }

    data_new.append(example_new)

with open(data_save_path, "w") as f:
    json.dump(data_new, f, indent=4)

In [17]:
from pymongo import MongoClient

# 创建MongoDB连接
client = MongoClient('mongodb://localhost:27017')  # 请替换为您的数据库地址、用户名、密码
db = client['customers_and_products_contacts']  # 替换为您的数据库名

# 执行聚合查询
pipeline = [
    {
        "$unwind": "$Customer_Orders"
    },
    {
        "$unwind": "$Customer_Orders.Order_Items"
    },
    {
        "$group": {
            "_id": "$customer_id",
            "total_order_quantity": { "$sum": { "$toInt": "$Customer_Orders.Order_Items.order_quantity" } }
        }
    },
    {
        "$sort": { "total_order_quantity": -1 }
    },
    {
        "$limit": 1
    },
    {
        "$lookup": {
            "from": "Customers",
            "localField": "_id",
            "foreignField": "customer_id",
            "as": "customer_info"
        }
    },
    {
        "$unwind": "$customer_info"
    },
    {
        "$project": {
            "_id": 0,
            "customer_name": "$customer_info.customer_name",
            "customer_phone": "$customer_info.customer_phone"
        }
    }
]

try:
    result = db['Customers'].aggregate(pipeline)
except Exception as ex:
    info = {"info":f"{ex}", "type":f"{type(ex)}"}

# # 打印结果
# for doc in result:
#     print(doc)

In [18]:
info

{'info': 'Failed to parse number \'male\' in $convert with no onError value: Bad digit "m" while parsing male, full error: {\'ok\': 0.0, \'errmsg\': \'Failed to parse number \\\'male\\\' in $convert with no onError value: Bad digit "m" while parsing male\', \'code\': 241, \'codeName\': \'ConversionFailure\'}',
 'type': "<class 'pymongo.errors.OperationFailure'>"}